# Extended model evaluation

Evaluation-only notebook (no training). Baseline metrics match `train_model.ipynb`, then three protocols from the scratch notebooks:

1. **Dual-threshold** — apply postprocess isolation rule to labeled chips using bulk-inference neighbors
2. **Tune `t_iso`** — sweep isolation threshold at fixed `t_main` with known ± regions
3. **Cumulative detections** — score chips by intersection with dual-threshold-filtered yearly detections accumulated through a chosen year

**Caveat (3):** inference patches and labeled chips are not the same grid; near-mine hard negatives can look like false positives when a chip only grazes a detection polygon.


## Setup


In [ ]:
from pathlib import Path
import sys

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import tensorflow as tf
from IPython.display import display
from shapely.geometry import box
from shapely.ops import unary_union
from shapely.prepared import prep
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    fbeta_score,
    precision_recall_fscore_support,
)
from sklearn.neighbors import NearestNeighbors

parent_dir = Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

import model_library
from postprocess import (
    KTH_NEIGHBOR_FIELD,
    dual_threshold_filter,
    keep_dual_threshold,
)

%load_ext autoreload
%autoreload 2

# --- paths / model (edit as needed) ---
data_timestamp = "2026-05-04T09:47"
data_dir = Path(f"../data/training_patches{data_timestamp}")
model_name = "48px_v4.10b-18d-20g-21a-22bc-ensemble"
model_path = Path(f"../models/{model_name}.h5")

output_dir = Path("../data/outputs/48px_v4.10b-18d-20g-21a-22bc-ensemble")
# Prefer raw detections under the ensemble folder when present:
inference_path = (
    output_dir / "raw_detections"
    / f"Amazon_ACA_{model_name}_0.40_2024-01-01_2024-12-31.geojson"
)
if not inference_path.is_file():
    inference_path = (
        output_dir / f"Amazon_ACA_{model_name}_0.40_2024-01-01_2024-12-31.geojson"
    )

known_negative_regions_path = Path("../data/boundaries/known_negative_regions.geojson")
known_positive_regions_path = Path("../data/boundaries/known_positive_regions.geojson")

EVAL_SPLITS = ["val", "test1", "test2", "test3"]
COMBO_SPLITS = ["val", "test2", "test3"]  # excludes test1
KN_SPLIT = "synthetic_known_negative"
KP_SPLIT = "synthetic_known_positive"


In [ ]:
def load_dataset(data_dir, split, bands_to_use=None, return_meta=False):
    """Load */{split}/{0,1}/*.tif into RAM. Optional chip footprints via return_meta."""
    root = Path(data_dir)
    files = sorted(root.glob(f"*/{split}/0/*.tif")) + sorted(root.glob(f"*/{split}/1/*.tif"))
    if not files:
        raise FileNotFoundError(f"No .tif under */{split}/{{0,1}}/*.tif in {root}")

    imgs, labels, paths, geoms = [], [], [], []
    for file_path in files:
        file_path = Path(file_path)
        with rasterio.open(file_path) as src:
            arr = src.read()
            if bands_to_use is not None:
                arr = arr[np.array(bands_to_use, dtype=np.intp), :, :]
            arr = np.moveaxis(arr, 0, -1).astype(np.float32) / 10000.0
            imgs.append(arr)
            if return_meta:
                paths.append(file_path.resolve().as_posix())
                geoms.append(box(*src.bounds))
        labels.append(int(file_path.parent.name))

    X = np.stack(imgs, axis=0)
    y = np.array(labels, dtype=np.int32)
    if return_meta:
        return X, y, {"path": paths, "geometry": geoms}
    return X, y


def flatten_binary_preds(preds):
    preds = np.asarray(preds)
    if preds.ndim == 2:
        if preds.shape[1] == 1:
            return preds.squeeze()
        if preds.shape[1] == 2:
            return preds[:, 1]
        return np.mean(preds, axis=1)
    return preds


def make_tf_dataset(X, y, batch_size=32):
    return (
        tf.data.Dataset.from_tensor_slices((X, y))
        .batch(batch_size)
        .prefetch(8)
    )


def report_table(y_true, y_pred, scores=None):
    report = classification_report(
        y_true, y_pred, target_names=["No Mine", "Mine"], output_dict=True
    )
    if scores is not None:
        print(f"Avg precision: {average_precision_score(y_true, scores):.4f}")
    df = pd.DataFrame(report).transpose()
    display(df.drop([i for i in df.index if "avg" in i]))
    return report


def kth_neighbor_km_query(query_coords_m, ref_coords_m, k):
    nq = len(query_coords_m)
    out = np.full(nq, np.nan, dtype=np.float64)
    if len(ref_coords_m) < k or nq == 0:
        return out
    nn = NearestNeighbors(n_neighbors=k, algorithm="kd_tree")
    nn.fit(ref_coords_m)
    dists_m, _ = nn.kneighbors(query_coords_m)
    out[:] = dists_m[:, k - 1] / 1000.0
    return out


def assign_kth_to_chips(chip_gdf, catalog_gdf, k=5):
    """k-th NN km from chip centroids to catalog; use k+1 if chip intersects catalog.

    Query and catalog centroids are projected into one shared metric CRS (from the
    catalog), so distances stay meaningful across geographic splits.
    """
    rows = chip_gdf.copy().reset_index(drop=True)
    cat = catalog_gdf.copy()
    if rows.crs is None:
        rows = rows.set_crs("EPSG:4326")
    if cat.crs is None:
        cat = cat.set_crs("EPSG:4326")
    cat = cat.to_crs(rows.crs)

    out = np.full(len(rows), np.nan, dtype=np.float64)
    if len(rows) == 0 or len(cat) == 0:
        rows[KTH_NEIGHBOR_FIELD] = out
        return rows

    try:
        metric_crs = cat.estimate_utm_crs()
    except Exception:
        metric_crs = "EPSG:3857"
    cat_m = cat.to_crs(metric_crs)
    rows_m = rows.to_crs(metric_crs)
    ref = np.column_stack(
        [cat_m.geometry.centroid.x.to_numpy(dtype=np.float64),
         cat_m.geometry.centroid.y.to_numpy(dtype=np.float64)]
    )
    query = np.column_stack(
        [rows_m.geometry.centroid.x.to_numpy(dtype=np.float64),
         rows_m.geometry.centroid.y.to_numpy(dtype=np.float64)]
    )

    overlap = np.zeros(len(rows), dtype=bool)
    hits = gpd.sjoin(rows[["geometry"]], cat[["geometry"]], how="inner", predicate="intersects")
    if len(hits):
        overlap[hits.index.to_numpy()] = True

    if (~overlap).any():
        out[~overlap] = kth_neighbor_km_query(query[~overlap], ref, k)
    if overlap.any():
        out[overlap] = kth_neighbor_km_query(query[overlap], ref, k + 1)

    rows[KTH_NEIGHBOR_FIELD] = out
    return rows


## Load model and score eval chips


In [ ]:
model = tf.keras.models.load_model(model_path, safe_mode=False)
model.summary()

batch_size = 32
bands_to_use = list(range(13))
if model.input_shape[-1] == 12:
    bands_to_use.remove(10)

parts = []
for split in EVAL_SPLITS:
    try:
        X, y, meta = load_dataset(data_dir, split, bands_to_use=bands_to_use, return_meta=True)
    except FileNotFoundError as exc:
        print(f"Skipping {split}: {exc}")
        continue
    ds = make_tf_dataset(X, y, batch_size=batch_size)
    with tf.device("/CPU:0"):
        scores = flatten_binary_preds(model.predict(ds, verbose=1))
    gdf = gpd.GeoDataFrame(
        {"y": y, "pred_score": scores, "split": split, "path": meta["path"]},
        geometry=meta["geometry"],
        crs="EPSG:4326",
    )
    parts.append(gdf)

if not parts:
    raise RuntimeError("No eval splits loaded; check data_dir.")
eval_gdf = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), crs="EPSG:4326")
if "val" not in set(eval_gdf["split"]):
    raise RuntimeError("val split required for curves.")

y_val = eval_gdf.loc[eval_gdf["split"] == "val", "y"].to_numpy()
preds_val = eval_gdf.loc[eval_gdf["split"] == "val", "pred_score"].to_numpy()
print(eval_gdf["split"].value_counts())


## Baseline metrics (common)


In [ ]:
def acc_curve(preds, y_true, thresholds=np.arange(0.01, 1.01, 0.01)):
    score = [np.mean((preds >= t).astype(int) == y_true) for t in thresholds]
    best = int(np.argmax(score))
    plt.plot(thresholds, score)
    plt.xlabel("Threshold")
    plt.ylabel("Accuracy")
    plt.title(f"Optimal t={thresholds[best]:.2f}  accuracy={score[best]:.2f}")


def fbeta_curve(preds, y_true, beta=1, thresholds=np.arange(0.01, 1.01, 0.01)):
    fbetas = [fbeta_score(y_true, preds >= t, beta=beta) for t in thresholds]
    best = int(np.argmax(fbetas))
    fig, ax = plt.subplots()
    ax.plot(thresholds, fbetas)
    ax.set_xlabel("Threshold")
    ax.set_ylabel(f"F{beta:g}")
    ax.set_title(f"Optimal t={thresholds[best]:.2f}  F{beta:g}={fbetas[best]:.2f}")
    return fig, ax


acc_curve(preds_val, y_val)
fbeta_curve(preds_val, y_val, beta=1)
fbeta_curve(preds_val, y_val, beta=0.5)


In [ ]:
threshold = 0.55  # edit
print(f"{model_name} @ single threshold t={threshold:g}")

for split in EVAL_SPLITS + ["combined"]:
    if split == "combined":
        sub = eval_gdf[eval_gdf["split"].isin(COMBO_SPLITS)]
        label = "combined-val-test2-test3"
    else:
        sub = eval_gdf[eval_gdf["split"] == split]
        label = split
    if sub.empty:
        continue
    print(label)
    report_table(sub["y"], sub["pred_score"] > threshold, scores=sub["pred_score"])


In [ ]:
threshold = 0.55
target_names = ["No Mine", "Mine"]
training_dataset = data_dir.stem

with open(model_path.with_suffix("").as_posix() + f"_config-t{threshold}.txt", "w") as f:
    f.write(f"Training dataset: {training_dataset}")
    f.write(f"\nBatch Size: {batch_size}")
    f.write(f"\n\nClassification reports at threshold {threshold}\n")
    for split in EVAL_SPLITS:
        sub = eval_gdf[eval_gdf["split"] == split]
        if sub.empty:
            continue
        f.write(f"\n--- {split} ---\n")
        f.write(classification_report(
            sub["y"],
            sub["pred_score"] > threshold,
            target_names=target_names,
        ))


## 1. Dual-threshold evaluation (bulk inference context)

Uses bulk-inference patches with `confidence >= t_main` as the neighbor catalog. Each labeled chip gets `kth_neighbor_km`. Isolated chips (`kth > D`) must also meet `t_iso`; clustered chips need only `t_main`. Label rule from `postprocess.keep_dual_threshold`.


In [ ]:
t_main = 0.55
t_iso = 0.80
k_neighbor = 5
isolation_km = 3.0

if not inference_path.is_file():
    raise FileNotFoundError(inference_path.resolve())

patches = gpd.read_file(inference_path)
if patches.crs is None:
    patches = patches.set_crs("EPSG:4326")

catalog = patches.loc[patches["confidence"].to_numpy(dtype=np.float64) >= t_main].copy()
print(f"{inference_path.name}: {len(patches)} patches, {len(catalog)} in catalog (>= {t_main:g})")

# Assign isolation distance to every labeled chip
chip_parts = []
for split, sub in eval_gdf.groupby("split"):
    if split not in EVAL_SPLITS:
        continue
    tagged = assign_kth_to_chips(sub, catalog, k=k_neighbor)
    chip_parts.append(tagged)
    kth = tagged[KTH_NEIGHBOR_FIELD].to_numpy()
    n_iso = int(np.sum(np.isfinite(kth) & (kth > isolation_km)))
    print(f"{split}: {len(tagged)} chips, {n_iso} isolated (>{isolation_km:g} km)")

eval_dual = gpd.GeoDataFrame(pd.concat(chip_parts, ignore_index=True), crs="EPSG:4326")

print(f"\n{model_name} dual @ t_main={t_main:g}, t_iso={t_iso:g}, k={k_neighbor}, D={isolation_km:g} km")
for split in EVAL_SPLITS + ["combined"]:
    if split == "combined":
        sub = eval_dual[eval_dual["split"].isin(COMBO_SPLITS)]
        label = "combined-val-test2-test3"
    else:
        sub = eval_dual[eval_dual["split"] == split]
        label = split
    if sub.empty:
        continue
    y_pred = keep_dual_threshold(
        sub["pred_score"].to_numpy(dtype=np.float64),
        sub[KTH_NEIGHBOR_FIELD].to_numpy(dtype=np.float64),
        t_main=t_main,
        t_iso=t_iso,
        isolation_km=isolation_km,
    )
    print(label)
    report_table(sub["y"], y_pred, scores=sub["pred_score"])


## 2. Tune `t_iso` at fixed `t_main`

Hold `t_main` fixed. Sweep `t_iso` on the **isolated** stratum (`kth > D`):

- left: recall of isolated positives (eval chips + optional known-positive regions)
- right: how many isolated negatives / known-negatives are suppressed

Synthetic KN/KP rows are inference patches intersecting `known_negative_regions.geojson` / `known_positive_regions.geojson`, with `pred_score = confidence`.


In [ ]:
t_main_fixed = 0.43
t_iso_grid = np.arange(t_main_fixed, 0.90, 0.01)

catalog_fixed = patches.loc[patches["confidence"].to_numpy(dtype=np.float64) >= t_main_fixed].copy()

chip_parts = [
    assign_kth_to_chips(sub, catalog_fixed, k=k_neighbor)
    for split, sub in eval_gdf.groupby("split")
    if split in EVAL_SPLITS
]
chips = gpd.GeoDataFrame(pd.concat(chip_parts, ignore_index=True), crs="EPSG:4326")


def patches_in_regions(region_path, y_value, split_name):
    """Inference patches intersecting known ± regions, scored by confidence."""
    if not region_path.is_file():
        print(f"Missing {region_path}; skipping {split_name}")
        return chips.iloc[:0].copy()
    regions = gpd.read_file(region_path).to_crs(patches.crs)
    hit = patches[patches.geometry.intersects(unary_union(regions.geometry.values))].copy()
    hit = hit.reset_index(drop=True)
    tagged = assign_kth_to_chips(
        gpd.GeoDataFrame(
            {
                "y": np.full(len(hit), y_value, dtype=np.int32),
                "pred_score": hit["confidence"].astype(np.float64),
                "split": split_name,
            },
            geometry=hit.geometry,
            crs=hit.crs,
        ),
        catalog_fixed,
        k=k_neighbor,
    )
    print(f"{split_name}: {len(tagged)} patches from {region_path.name}")
    return tagged


kn = patches_in_regions(known_negative_regions_path, 0, KN_SPLIT)
kp = patches_in_regions(known_positive_regions_path, 1, KP_SPLIT)

iso_chip = chips[KTH_NEIGHBOR_FIELD].to_numpy(dtype=np.float64) > isolation_km
chip_iso_pos = chips.loc[iso_chip & (chips["y"] == 1)]
chip_iso_neg = chips.loc[iso_chip & (chips["y"] == 0)]
kp_iso = kp.loc[kp[KTH_NEIGHBOR_FIELD].to_numpy(dtype=np.float64) > isolation_km] if len(kp) else kp

kn_at_main = kn["pred_score"].to_numpy(dtype=np.float64) >= t_main_fixed if len(kn) else np.array([], dtype=bool)
neg_at_main = chip_iso_neg["pred_score"].to_numpy(dtype=np.float64) >= t_main_fixed

rows = []
for t_iso in t_iso_grid:
    row = {"t_iso": float(t_iso)}
    if len(chip_iso_pos):
        hat = keep_dual_threshold(
            chip_iso_pos["pred_score"].to_numpy(dtype=np.float64),
            chip_iso_pos[KTH_NEIGHBOR_FIELD].to_numpy(dtype=np.float64),
            t_main=t_main_fixed, t_iso=float(t_iso), isolation_km=isolation_km,
        )
        row["recall_chip_iso_pos"] = hat.mean()
    else:
        row["recall_chip_iso_pos"] = np.nan

    if len(kp_iso):
        hat = keep_dual_threshold(
            kp_iso["pred_score"].to_numpy(dtype=np.float64),
            kp_iso[KTH_NEIGHBOR_FIELD].to_numpy(dtype=np.float64),
            t_main=t_main_fixed, t_iso=float(t_iso), isolation_km=isolation_km,
        )
        row["recall_kp_iso_pos"] = hat.mean()
    else:
        row["recall_kp_iso_pos"] = np.nan

    if len(kn):
        hat = keep_dual_threshold(
            kn["pred_score"].to_numpy(dtype=np.float64),
            kn[KTH_NEIGHBOR_FIELD].to_numpy(dtype=np.float64),
            t_main=t_main_fixed, t_iso=float(t_iso), isolation_km=isolation_km,
        )
        row["kn_remain_frac"] = hat.mean()
        row["kn_removed_at_main"] = int((kn_at_main & ~hat).sum())
    else:
        row["kn_remain_frac"] = np.nan
        row["kn_removed_at_main"] = 0

    if neg_at_main.any():
        hat = keep_dual_threshold(
            chip_iso_neg["pred_score"].to_numpy(dtype=np.float64),
            chip_iso_neg[KTH_NEIGHBOR_FIELD].to_numpy(dtype=np.float64),
            t_main=t_main_fixed, t_iso=float(t_iso), isolation_km=isolation_km,
        )
        row["eval_neg_iso_remain_frac"] = (neg_at_main & hat).sum() / neg_at_main.sum()
    else:
        row["eval_neg_iso_remain_frac"] = np.nan

    rows.append(row)

iso_sweep = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
ax = axes[0]
ax.plot(iso_sweep["t_iso"], iso_sweep["recall_chip_iso_pos"], marker="o",
        label=f"eval iso y=1 (n={len(chip_iso_pos)})")
if iso_sweep["recall_kp_iso_pos"].notna().any():
    ax.plot(iso_sweep["t_iso"], iso_sweep["recall_kp_iso_pos"], marker="^",
            label=f"KP iso y=1 (n={len(kp_iso)})")
ax.set_ylim(0, 1.05)
ax.set_xlabel(r"$t_{iso}$")
ax.set_ylabel("Recall")
ax.set_title(f"Isolated positives @ t={t_main_fixed:g}")
ax.legend(fontsize=8)

ax = axes[1]
ax.plot(iso_sweep["t_iso"], iso_sweep["kn_remain_frac"], marker="o", label="KN remain frac")
ax.plot(iso_sweep["t_iso"], iso_sweep["eval_neg_iso_remain_frac"], marker="s",
        label="eval iso neg remain (of those ≥ t_main)")
ax.set_ylim(0, 1.05)
ax.set_xlabel(r"$t_{iso}$")
ax.set_ylabel("Fraction remaining")
ax.set_title("Negative suppression")
ax.legend(fontsize=8)
plt.show()

#display(iso_sweep.round(4))


## 3. Cumulative detections vs labeled chips

Matches the published cumulative product more closely than a raw `confidence >= t` union:

1. Load yearly Amazon raw detections through the latest `cumul_through_years` end year.
2. At each threshold `t`, run **per-year** `dual_threshold_filter` (`t_main=t`, `t_iso=t+offset`, same k/D as postprocess), for each through-year (default **2022** and **2025**).
3. Optionally append Andes supplemental (`confidence >= t_andes`).
4. Predict chip positive via **spatial join** (intersects any kept patch) — not a single `union_all()` geometry.

Plots are side-by-side for each through-year. A raw-only union over-calls vs the QGIS `cumulative_t0.55_…` layer, because isolated false positives survive single-thresholding but are dropped by the dual-threshold spatial prior.

**Limitations:** 

* Bulk-inference patches and training/eval chips are different grids. A true hard-negative chip near a mine can intersect a detection polygon and look like an FP; conversely a positive chip may miss the union. Treat metrics as approximate operational proxies, not pure classifier scores.

* Training data was collected for 2019 and then at various intervals through 2023-2025. Labels may not reflect the state of mining in earlier years (mine pits not yet opened). 


In [ ]:
# Cumulative through-years (inclusive), dual-thresholded like production
cumul_through_years = [2022, 2025]
cumul_thresholds = np.round(np.arange(0.40, 0.80, 0.05), 2)
cumul_splits = COMBO_SPLITS

# Dual-threshold params (defaults match website cumulatives at t_main=0.55 → t_iso=0.80)
k_cumul = 5
isolation_km_cumul = 3.0
t_iso_offset = 0.25  # t_iso = min(0.99, t_main + t_iso_offset)
include_andes = False
t_andes = 0.2


def year_raw_path(year):
    p = (
        output_dir / "raw_detections"
        / f"Amazon_ACA_{model_name}_0.40_{year}-01-01_{year}-12-31.geojson"
    )
    return p if p.is_file() else output_dir / p.name


def year_andes_path(year):
    p = (
        output_dir / "raw_detections" / "andes_supplemental"
        / f"andes_supplemental_{model_name}_0.2_{year}-01-01_{year}-12-31.geojson"
    )
    if not p.is_file():
        alt = p.with_name(p.name.replace("_0.2_", "_0.20_"))
        if alt.is_file():
            return alt
    return p


def read_detection_geojson(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path.resolve())
    gdf = gpd.read_file(path)
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")
    if "confidence" not in gdf.columns:
        gdf["confidence"] = 1.0
    return gdf[gdf.geometry.notnull() & ~gdf.geometry.is_empty].copy()


# Load yearly raw once through the latest end year
all_years = list(range(2018, max(cumul_through_years) + 1))
yearly_amazon = {y: read_detection_geojson(year_raw_path(y)) for y in all_years}
print({y: len(g) for y, g in yearly_amazon.items()})
yearly_andes = {}
if include_andes:
    for y in all_years:
        ap = year_andes_path(y)
        if ap.is_file():
            yearly_andes[y] = read_detection_geojson(ap)
            print(f"andes {y}: {len(yearly_andes[y])}")
        else:
            print(f"andes {y}: missing ({ap.name})")


def cumulative_patches_at_threshold(t_main, through_year):
    """Per-year dual-threshold Amazon (+ optional Andes) through through_year."""
    t_iso = float(min(0.99, t_main + t_iso_offset))
    years = list(range(2018, through_year + 1))
    parts = [
        dual_threshold_filter(
            yearly_amazon[y],
            t_main=float(t_main),
            t_iso=t_iso,
            k=k_cumul,
            isolation_km=isolation_km_cumul,
        )
        for y in years
    ]
    for y in years:
        if y in yearly_andes:
            g = yearly_andes[y]
            parts.append(g.loc[g["confidence"].to_numpy(dtype=np.float64) >= t_andes].copy())
    parts = [p for p in parts if len(p)]
    if not parts:
        return gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")
    return gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), crs=parts[0].crs)


def chip_hits_patches(chip_gdf, patches_gdf):
    """Boolean ndarray aligned to chip_gdf rows: intersects any patch."""
    if chip_gdf.empty or patches_gdf.empty:
        return np.zeros(len(chip_gdf), dtype=bool)
    hits = gpd.sjoin(
        chip_gdf[["geometry"]],
        patches_gdf[["geometry"]],
        how="inner",
        predicate="intersects",
    )
    return chip_gdf.index.isin(hits.index.unique())


rows = []
cumul_patches_by_t = {}  # (through_year, t) -> patches
for through_year in cumul_through_years:
    for t in cumul_thresholds:
        patches = cumulative_patches_at_threshold(t, through_year)
        cumul_patches_by_t[(through_year, float(t))] = patches
        print(
            f"through {through_year}, t={t:g}: {len(patches)} patches "
            f"(+andes={include_andes})"
        )
        for split in cumul_splits:
            sub = eval_gdf[eval_gdf["split"] == split]
            if sub.empty:
                continue
            y_pred = chip_hits_patches(sub, patches).astype(np.uint8)
            p, r, f1, _ = precision_recall_fscore_support(
                sub["y"].to_numpy(), y_pred, average="binary", zero_division=0
            )
            rows.append({
                "through_year": through_year,
                "split": split,
                "threshold": float(t),
                "n_detections_kept": len(patches),
                "precision": float(p),
                "recall": float(r),
                "f1": float(f1),
            })

cumul_df = pd.DataFrame(rows)
best = (
    cumul_df.loc[cumul_df.groupby(["through_year", "split"])["f1"].idxmax()]
    .sort_values(["through_year", "split"])
)
display(best.reset_index(drop=True))

for split in cumul_splits:
    if split not in set(cumul_df["split"]):
        continue
    fig, axes = plt.subplots(
        1, len(cumul_through_years), figsize=(6 * len(cumul_through_years), 4),
        sharey=True, constrained_layout=True,
    )
    if len(cumul_through_years) == 1:
        axes = [axes]
    for ax, through_year in zip(axes, cumul_through_years):
        part = (
            cumul_df[
                (cumul_df["split"] == split)
                & (cumul_df["through_year"] == through_year)
            ].sort_values("threshold")
        )
        ax.plot(part["threshold"], part["precision"], marker="o", label="precision")
        ax.plot(part["threshold"], part["recall"], marker="o", label="recall")
        ax.plot(part["threshold"], part["f1"], marker="o", label="f1")
        ax.set_title(f"{split}: cumulative through {through_year}")
        ax.set_xlabel("t_main")
        ax.set_ylim(0, 1.05)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)
    axes[0].set_ylabel("score")
    plt.show()


In [ ]:
# Per-class metrics at a chosen t_main (mine = 1, no-mine = 0)
cumul_t_report = 0.55  # edit

for through_year in cumul_through_years:
    key = (through_year, float(cumul_t_report))
    patches = cumul_patches_by_t.get(key)
    if patches is None:
        patches = cumulative_patches_at_threshold(cumul_t_report, through_year)
    t_iso_r = float(min(0.99, cumul_t_report + t_iso_offset))
    print(
        f"\n=== Cumulative through {through_year} @ t_main={cumul_t_report:g} "
        f"(t_iso={t_iso_r:g}, n_patches={len(patches)}) ==="
    )
    for split in cumul_splits:
        sub = eval_gdf[eval_gdf["split"] == split]
        if sub.empty:
            continue
        y_pred = chip_hits_patches(sub, patches).astype(np.uint8)
        print(split)
        report_table(sub["y"], y_pred)


In [ ]:
# Export eval chips with cumulative-protocol labels at a fixed threshold / through-year.
# Requires the cumulative cell above.
cumul_t_export = 0.55
cumul_through_year_export = 2025  # or 2022
export_splits = cumul_splits

out = eval_gdf[eval_gdf["split"].isin(export_splits)].copy()
key = (cumul_through_year_export, float(cumul_t_export))
patches = cumul_patches_by_t.get(key)
if patches is None:
    patches = cumulative_patches_at_threshold(cumul_t_export, cumul_through_year_export)
out["cumul_pred"] = chip_hits_patches(out, patches).astype(np.uint8)
out["cumul_threshold"] = cumul_t_export
out["cumul_through_year"] = cumul_through_year_export
out["t_iso"] = float(min(0.99, cumul_t_export + t_iso_offset))

export_path = (
    output_dir
    / (
        f"eval_chips_cumul{cumul_through_year_export}"
        f"_t{cumul_t_export:g}_dual_{model_name}.geojson"
    )
)
export_path.parent.mkdir(parents=True, exist_ok=True)
out.to_file(export_path, driver="GeoJSON")
print(
    f"Wrote {len(out)} chips ({len(patches)} patches in cumulative) -> {export_path.resolve()}"
)
print(f"columns: {list(out.columns)}")
print(out[["y", "cumul_pred"]].value_counts().sort_index())
